In [4]:
library(readxl)
library(dplyr)
library(ggplot2)
library(tidyr)
library(ggrepel)
library(networkD3)
library(tibble)
library(ggalluvial)
library(RColorBrewer)
library(scales)
library(viridis)
library(dplyr)
library(readr)


Attaching package: ‘readr’


The following object is masked from ‘package:scales’:

    col_factor




In [2]:
cytokine <- read.csv("/vol/projects/yzhang/500FG_aging/input/cytokine/filtered_cytokine_filled_trans.csv")
rownames(cytokine) <- cytokine[,1]
cytokine <- cytokine[,-1]
head(cytokine)
dim(cytokine) #489samples 91 cytokines
sum(is.na(cytokine))

,IFNy_LPS_WB_48h,IFNy_PHA_WB_48h,IFNy_C.conidiaHK_WB_48h,IFNy_S.aureus_WB_48h,IL1b_LPS_WB_48h,IL1b_PHA_WB_48h,IL1b_C.conidiaHK_WB_48h,IL1b_S.aureus_WB_48h,IL6_LPS_WB_48h,IL6_PHA_WB_48h,⋯,IL22_Cryptococcus_PBMC_7days,IL22_Bacteroides_PBMC_7days,IL6_LPS_macroPG_24h,IL6_C.albicansconidia_macroPG_24h,IL6_MTB_macroPG_24h,IL6_S.typhimurium_macroPG_24h,TNFA_LPS_macroPG_24h,TNFA_C.albicansconidia_macroPG_24h,TNFA_MTB_macroPG_24h,TNFA_S.typhimurium_macroPG_24h
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
HV001,10.828930,10.710806,8.965784,10.366322,11.34541,10.595257,8.392317,10.051209,13.26033,13.03480,⋯,NA,8.290019,8.335390,7.936638,8.005625,10.152285,7.459432,8.326429,6.285402,9.417853
HV002,9.457381,8.761551,9.511753,9.651052,11.43359,9.652845,8.558421,9.162391,13.74431,12.81238,⋯,NA,10.594325,10.156083,9.703904,9.142107,4.954196,8.927778,9.579316,7.149747,6.285402
HV003,8.413628,10.884934,5.857981,8.442943,11.03480,10.098032,7.475733,9.321928,12.89045,12.62708,⋯,NA,11.640697,8.467606,8.857981,8.927778,10.637531,9.592457,10.224002,8.654636,11.734710
HV004,5.643856,5.285402,5.285402,5.426265,10.69697,8.894818,10.008429,9.238405,13.48533,13.11586,⋯,NA,11.611486,8.224002,7.851749,9.079485,10.795228,8.129283,8.654636,7.781360,10.781360
HV005,8.957102,8.179909,5.700440,7.459432,10.68299,9.105909,7.475733,9.396605,13.04405,12.66977,⋯,NA,NA,7.845490,6.149747,7.864186,9.829723,6.285402,7.149747,6.507795,9.529431
HV006,8.774787,6.727920,5.781360,7.483816,10.22641,7.599913,6.266787,8.761551,12.75572,12.06205,⋯,NA,10.087463,7.876517,7.303781,8.921841,10.710806,7.930737,7.375039,7.149747,9.946906


[1] 489  91

[1] 2029

In [3]:
basicPhenos = read.csv("/vol/projects/yzhang/500FG_aging/output/02_Olink/Age_group_basicPhenos.csv",row.names=1)
rownames(basicPhenos) = basicPhenos$ID_500fg
## Filter a feature matrix and phenotype table to shared sample IDs and align rows in the same order
prep_filter_match <- function(feature_mat, basicPhenos) {
  idx <- intersect(rownames(feature_mat), rownames(basicPhenos))
  feature_filter <- feature_mat[idx, , drop = FALSE]
  pheno_match <- basicPhenos[rownames(feature_filter), , drop = FALSE]
  list(feature_filter = feature_filter, basicPhenos_filter_match = pheno_match)
}
# usage
res <- prep_filter_match(cytokine, basicPhenos)
cytokine_filter <- res$feature_filter
basicPhenos_filter_match <- res$basicPhenos_filter_match
##one missing gender value
basicPhenos_filter_match$Gender <- trimws(basicPhenos_filter_match$Gender)
basicPhenos_filter_match$Gender[basicPhenos_filter_match$Gender == ""] <- NA

head(cytokine_filter)
dim(cytokine_filter)
head(basicPhenos_filter_match)
dim(basicPhenos_filter_match)

,IFNy_LPS_WB_48h,IFNy_PHA_WB_48h,IFNy_C.conidiaHK_WB_48h,IFNy_S.aureus_WB_48h,IL1b_LPS_WB_48h,IL1b_PHA_WB_48h,IL1b_C.conidiaHK_WB_48h,IL1b_S.aureus_WB_48h,IL6_LPS_WB_48h,IL6_PHA_WB_48h,⋯,IL22_Cryptococcus_PBMC_7days,IL22_Bacteroides_PBMC_7days,IL6_LPS_macroPG_24h,IL6_C.albicansconidia_macroPG_24h,IL6_MTB_macroPG_24h,IL6_S.typhimurium_macroPG_24h,TNFA_LPS_macroPG_24h,TNFA_C.albicansconidia_macroPG_24h,TNFA_MTB_macroPG_24h,TNFA_S.typhimurium_macroPG_24h
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
HV001,10.828930,10.710806,8.965784,10.366322,11.34541,10.595257,8.392317,10.051209,13.26033,13.03480,⋯,NA,8.290019,8.335390,7.936638,8.005625,10.152285,7.459432,8.326429,6.285402,9.417853
HV002,9.457381,8.761551,9.511753,9.651052,11.43359,9.652845,8.558421,9.162391,13.74431,12.81238,⋯,NA,10.594325,10.156083,9.703904,9.142107,4.954196,8.927778,9.579316,7.149747,6.285402
HV003,8.413628,10.884934,5.857981,8.442943,11.03480,10.098032,7.475733,9.321928,12.89045,12.62708,⋯,NA,11.640697,8.467606,8.857981,8.927778,10.637531,9.592457,10.224002,8.654636,11.734710
HV004,5.643856,5.285402,5.285402,5.426265,10.69697,8.894818,10.008429,9.238405,13.48533,13.11586,⋯,NA,11.611486,8.224002,7.851749,9.079485,10.795228,8.129283,8.654636,7.781360,10.781360
HV005,8.957102,8.179909,5.700440,7.459432,10.68299,9.105909,7.475733,9.396605,13.04405,12.66977,⋯,NA,NA,7.845490,6.149747,7.864186,9.829723,6.285402,7.149747,6.507795,9.529431
HV006,8.774787,6.727920,5.781360,7.483816,10.22641,7.599913,6.266787,8.761551,12.75572,12.06205,⋯,NA,10.087463,7.876517,7.303781,8.921841,10.710806,7.930737,7.375039,7.149747,9.946906


[1] 482  91

,ID_500fg,Gender,Age,Height,Weight,BMI,LivingSituation,Education,Ethnicity,FarmAnimal,Pets,Travel,DayInMonth,monthinYear,dayInYear,year,AgeGroup
,<chr>,<chr>,<int>,<int>,<int>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<chr>
HV001,HV001,male,27,178,70,22.09317,Married,University degree,European,Once a month,Yes,No,24,7,204,2013,Group 5
HV002,HV002,male,26,178,68,21.46194,Cohabitation,University degree,European,Rarely,No,Yes,24,7,204,2013,Group 4
HV003,HV003,female,24,173,68,22.72044,Married,University degree,European,Rarely,Yes,Yes,26,8,236,2013,Group 4
HV004,HV004,female,20,173,65,21.71807,Married,University degree,European,Rarely,Yes,No,26,8,236,2013,Group 2
HV005,HV005,female,22,178,64,20.19947,Married,University student,European,Once a month,Yes,Yes,5,9,245,2013,Group 3
HV006,HV006,female,24,168,58,20.54989,Married,University degree,European,Rarely,Yes,No,NA,NA,NA,NA,Group 4


[1] 482  17

In [4]:
# model 1: age + cytokine (age not scaled) + Gender
all_results <- list()

for (i in seq_len(ncol(cytokine_filter))) {

  # Build a minimal modeling table
  data <- data.frame(
    feature = log(cytokine_filter[, i] + 1),
    age = basicPhenos_filter_match$Age,
    Gender = factor(basicPhenos_filter_match$Gender)  # convert chr to factor
  )

  # Linear regression
  mod <- lm(feature ~ age + Gender, data = data)

  # Extract coefficient and p-value for age (not Gender)
  cf <- summary(mod)$coefficients
  res <- cf["age", c("Estimate", "Pr(>|t|)")]

  # Store results
  result <- data.frame(
    feature  = colnames(cytokine_filter)[i],
    estimate = res["Estimate"],
    p        = res["Pr(>|t|)"]
  )

  # Signed -log10(p)
  result$value <- -log10(result$p) * result$estimate

  all_results[[i]] <- result
}

final_result <- do.call(rbind, all_results)
final_result$padj <- p.adjust(final_result$p, method = "BH")
final_result$sig <- ifelse(final_result$padj < 0.05, "sig", "no")

head(final_result)
dim(final_result)
table(final_result$sig)

write.csv(final_result, file = "/vol/projects/yzhang/500FG_aging/output/01_cytokine/cytokine_age_liner_gender_FDR.csv")

,feature,estimate,p,value,padj,sig
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
Estimate,IFNy_LPS_WB_48h,-1.489589e-03,0.0157429534,-2.685600e-03,0.057304350,no
Estimate1,IFNy_PHA_WB_48h,-1.248407e-03,0.1019912047,-1.237717e-03,0.257610692,no
Estimate2,IFNy_C.conidiaHK_WB_48h,-2.449299e-03,0.0004847994,-8.118050e-03,0.002757296,sig
Estimate3,IFNy_S.aureus_WB_48h,-1.828313e-03,0.0112181557,-3.565355e-03,0.050387503,no
Estimate4,IL1b_LPS_WB_48h,1.086956e-04,0.5573738995,2.759275e-05,0.768500377,no
Estimate5,IL1b_PHA_WB_48h,-9.867128e-05,0.8021676676,-9.446280e-06,0.912465722,no


[1] 91  6


 no sig 
 72  19 

In [6]:
save.image("/vol/projects/yzhang/500FG_aging/output/01_cytokine/cytokine_gender.RData")

In [2]:
load("/vol/projects/yzhang/500FG_aging/output/01_cytokine/cytokine_gender.RData")

In [7]:
final_result_sig <- final_result %>%
  filter(sig == "sig") %>%
  select(feature, estimate, p, padj)
head(final_result_sig)
dim(final_result_sig)
write_excel_csv(final_result_sig, file = "/vol/projects/yzhang/500FG_aging/output/01_cytokine/cytokine_age_liner_gender_FDR_sig.csv")

,feature,estimate,p,padj
,<chr>,<dbl>,<dbl>,<dbl>
Estimate2,IFNy_C.conidiaHK_WB_48h,-0.0024492990,4.847994e-04,2.757296e-03
Estimate9,IL6_PHA_WB_48h,-0.0004435195,1.039522e-02,4.978761e-02
Estimate42,IL6_C.albicanshyphae_PBMC_24h,0.0013671880,4.366248e-04,2.648857e-03
Estimate43,IL6_A.fumigatusconidiaSerum_PBMC_24h,0.0016975393,4.108165e-03,2.076906e-02
Estimate60,IFNy_B.burgdorferi_PBMC_7days,-0.0038129075,2.495544e-10,7.569818e-09
Estimate61,IFNy_Borreliamix_PBMC_7days,-0.0027042980,8.205513e-08,1.264237e-06


[1] 19  4